# Live testing scratchpad

Interactive testing of the live/streaming path (`streaming.py`, `live.py`)
against real and simulated LSL sources, without a real headset. Four
mostly-independent sections, run top to bottom or jump to whichever one
you need -- each says what it needs from earlier cells:

1. **Running a script in the background** -- the `BackgroundScript` helper
   every later section reuses to launch `scripts/*.py` CLI tools without
   blocking the kernel. Its own demo cells (outlet/consumer/trigger) are
   just that: a demo. Nothing later depends on having actually run them.
2. **Test words for build_curated_word.py** / **Spell a word via
   subprocess** -- quick sanity checks of the word-curation tool alone,
   no LSL involved.
3. **Session runner: build_word -> LiveDecoder -> diff** -- the real
   decode-correctness test: assembles a message from curated real
   epochs and runs it through an actual `LiveDecoder`, no LSL/subprocess
   involved either. Needs a trained model on disk (`MODEL_PATH` below).
4. **Live Test Segments B and C** -- the opposite kind of test: genuinely
   live LSL plumbing (real wall-clock marker/EEG timing across two
   processes), but with no real P300 in the replayed EEG, so these prove
   the streaming code doesn't crash/drop epochs, not that decoding is
   correct. Uses `BackgroundScript` from section 1.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import mne
import pylsl
import time

from eyecando.ingestion.data import load_moabb_all_subjects, load_moabb_session
from eyecando.pipeline.preprocessing import preprocess_p300
from eyecando.pipeline.spatial import XDawnFilter
from eyecando.pipeline.lda import LDAClassifier
from eyecando.live.streaming import LSLEEGStream, iter_epochs

## Running a script in the background

The scripts under `scripts/` (`simulate_lsl_outlet.py`, `run_live_session.py`,
`trigger_lsl_start.py`) are standalone CLI processes -- they block until
they finish or are stopped. Launched directly, any one of them would hold
up this notebook's kernel for as long as it runs, so the cell below spawns
one as a subprocess and drains its output on a background thread instead:
the launching cell returns immediately, the process's stdout/stderr stream
into the *next* cell's output as they arrive, and later cells (e.g.
`LSLEEGStream`/`iter_epochs`, or firing a trigger) can run in the same
kernel while it's still going.

A real OS thread (not `asyncio`) is used deliberately -- Jupyter's own
event loop already owns `asyncio` in this kernel, and a background
`asyncio` task would only actually run between cells, not while the
kernel is otherwise idle waiting on you. `subprocess.Popen` also means
`.stop()` can actually terminate the process; a plain in-process
`threading.Thread(target=some_scripts_main)` would run in the *same*
process as the kernel, sharing its memory and unable to be killed
independently if it hangs.

In [ ]:
import subprocess
import sys
import threading
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent  # scripts/simulate_live/ -> repo root, where scripts/ lives


class BackgroundScript:
    """Runs one of scripts/*.py as a subprocess without blocking the kernel.

    Starts the process immediately; a daemon thread drains its combined
    stdout/stderr line by line, both printing each one (prefixed with
    this script's name) as it arrives *and* appending it to self.lines.
    The live print only actually shows up if some cell happens to be
    running at that instant -- Jupyter has nowhere to attach output
    otherwise, which is why a background process's prints can seem to
    "go missing" between cells. self.lines doesn't have that problem:
    call .output() any time, in any later cell, to see everything
    printed so far, not just whatever arrived while a cell was open.

    The launching cell itself returns as soon as the process starts,
    leaving the kernel free for other cells.

    A real subprocess (not an in-kernel thread calling the script's own
    main()) so stop() can actually terminate it -- these scripts run
    indefinitely (simulate_lsl_outlet.py --wait-for-trigger) or hold an
    LSL connection open (run_live_session.py) with no other way to
    interrupt them from a notebook cell.

    start_new_session=True matters more than it looks: without it, the
    child inherits this process's *process group*, so interrupting the
    kernel (Jupyter's "Interrupt Kernel", or a stray Ctrl-C reaching this
    terminal) sends SIGINT to that whole group -- including the
    "background" subprocess, killing it via a plain KeyboardInterrupt
    right in the middle of whatever it was doing (confirmed directly:
    without this flag, os.getpgid(child.pid) == os.getpgid(os.getpid());
    with it, the child gets its own group and is immune). That defeats
    the entire point of backgrounding it.
    """

    def __init__(self, args: list[str], cwd: Path = REPO_ROOT) -> None:
        self.name = Path(args[0]).name
        self.lines: list[str] = []
        self.proc = subprocess.Popen(
            [sys.executable, *args],
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            start_new_session=True,
        )
        self._thread = threading.Thread(target=self._drain_output, daemon=True)
        self._thread.start()

    def _drain_output(self) -> None:
        assert self.proc.stdout is not None
        for line in self.proc.stdout:
            self.lines.append(line)
            print(f"[{self.name}] {line}", end="")

    def output(self, last_n: int | None = None) -> str:
        """Everything printed so far, or just the last `last_n` lines."""
        lines = self.lines if last_n is None else self.lines[-last_n:]
        return "".join(lines)

    def is_running(self) -> bool:
        return self.proc.poll() is None

    def stop(self, timeout: float = 5.0) -> None:
        """Terminate the process and wait for its output thread to drain."""
        if self.is_running():
            self.proc.terminate()
            try:
                self.proc.wait(timeout=timeout)
            except subprocess.TimeoutExpired:
                self.proc.kill()
                self.proc.wait(timeout=timeout)
        self._thread.join(timeout=timeout)

In [ ]:
# Example: launch simulate_lsl_outlet.py in the background. This cell
# returns immediately -- the outlet's own prints (LSL init, "streaming...",
# etc.) will appear as output on whichever cell is running when they
# arrive, since they're coming from the drain thread above, not this cell.
outlet = BackgroundScript(
    [
        "scripts/simulate_live/simulate_lsl_outlet.py",
        "--subject", "1",
        "--session", "0",
        "--wait-for-trigger",
    ]
)
print(f'Outlet Running: {outlet.is_running()}')
consumer = BackgroundScript(
    [
        "scripts/simulate_live/consume_lsl_stream.py",
        "--subject", "1",
        "--session", "0",
        "--wait-for-trigger",
    ]
)
print(f'Consumer Running: {consumer.is_running()}')
time.sleep(20)

In [ ]:
# Pull up everything each process has printed so far, any time, in any
# cell -- this doesn't depend on a cell having been open while the lines
# actually arrived (that's what live printing depends on; this doesn't).
print("--- outlet ---")
print(outlet.output())
print("--- consumer ---")
print(consumer.output())

# Just the most recent few lines, e.g. to check current status without
# re-printing everything from the start:
# outlet.output(last_n=5)

In [ ]:
# trigger_lsl_start.py just pushes one sample and exits -- a couple
# seconds at most -- so unlike the outlet above, there's no need to
# background it. subprocess.run() blocks until it exits and hands back
# a CompletedProcess with everything it printed; the kernel is free again
# as soon as this cell finishes.
result = subprocess.run(
    [sys.executable, "scripts/simulate_live/trigger_lsl_start.py"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout, end="")
print(result.stderr, end="")
result.check_returncode()  # raises if trigger_lsl_start.py didn't exit 0

In [ ]:
outlet.is_running(), consumer.is_running()

In [ ]:
# Other cells (LSLEEGStream, trigger_lsl_start.py via another BackgroundScript,
# etc.) can run here while `outlet` is still going.

# When done with it:
# outlet.stop()
# consumer.stop()
# outlet.is_running(), consumer.is_running()

## Test words for build_curated_word.py

**Requires:** nothing above -- self-contained (imports `build_word`
itself). Skip straight here if you just want to sanity-check curation.

A quick list of words/phrases to exercise `scripts/curate/build_curated_word.py`'s
minimal-session assignment -- only uppercase letters, digits 1-9, and space
are valid (that's `FARWELL_DONCHIN_GRID`'s full 36-symbol alphabet; no
lowercase, no punctuation, no `0`).

Mostly kept to words needing 5 or fewer distinct sessions -- verified
directly that `_minimal_session_assignment`'s exhaustive subset search
gets expensive once a word needs many distinct sessions from this
30-session corpus (its docstring already flags this as only tractable for
a corpus this size, but the practical cutoff is narrower than that
sounds): `MISSISSIPPI` (7 sessions, heavy repeats) took ~8.6s, and
`BRAIN COMPUTER INTERFACE` (7 sessions) took ~32s -- both still correct,
just slow. `MISSISSIPPI` is kept below as a deliberate repeat-heavy
stress test (expect a several-second delay on that one specifically);
nothing needing 7+ sessions is worth adding here until that search gets a
real set-cover solver instead of brute-force subset enumeration.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts" / "curate"))
from build_curated_word import build_word

TEST_WORDS = [
    "CAT",
    "HELLO",
    "TEST WORD",
    "GOOD JOB",
    "AGENT 47",
    "ROOM 237",
    "HELLO WORLD",
    "GOOD MORNING",
    "DATA SCIENCE",
    "NEURAL SIGNAL",
    "SPEAK YOUR MIND",
    "EYE CAN DO",
    "TYPE FAST TODAY",
    "MISSISSIPPI",  # repeat-heavy stress test -- expect a several-second delay
]

for word in TEST_WORDS:
    t0 = time.time()
    letters = build_word(word)
    n_sessions = len({(l["subject"], l["session"]) for l in letters})
    n_unwhitened = sum(1 for l in letters if l["aligner"] is None)
    print(
        f"{word!r}: {len(letters)} letters, {n_sessions} distinct sessions, "
        f"{n_unwhitened} without an aligner yet ({time.time() - t0:.2f}s)"
    )

## Spell a word via subprocess

Runs `scripts/curate/build_curated_word.py` as a standalone process (like the
LSL scripts above) instead of importing `build_word()` directly -- this
is the same CLI a real invocation would use. `.upper()` matters: the
grid is uppercase-only (see the alphabet note above), and a lowercase
word would fail `_grid_position`'s lookup (FARWELL_DONCHIN_GRID.index() is case-sensitive).

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent  # scripts/simulate_live/ -> repo root, where scripts/ lives

word = "hello world"  # edit this to whatever you want to spell

args = [
    sys.executable,
    str(REPO_ROOT / "scripts" / "curate" / "build_curated_word.py"),
    "--word", f"{word.upper()}",
]
result = subprocess.run(args, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(f"--- stderr ---\n{result.stderr}")

## Session runner: build_word -> LiveDecoder -> diff

**Requires:** nothing above (self-contained, imports `build_word` fresh)
except a real trained model file at `MODEL_PATH` on disk -- update that
path below if `models/p300_classifier.pkl` doesn't exist yet. Does NOT
need any of the LSL/`BackgroundScript` cells above; this section replays
already-curated epochs directly through `PrebuiltEpochStream`, no live
connection at all.

The piece that actually closes the loop: assemble a message with
`build_word()`, replay each letter's own curated epochs through a real
`LiveDecoder` via `PrebuiltEpochStream` (using that letter's own
already-fit aligner -- see build_curated_word.py's docstring for why
every letter gets its own), and diff the decoded text against what was
intended.

Two things this needs to get right, both found by actually running this
and reading the resulting log instead of trusting a bare pass/fail:

1. **Permutation-aware decoding.** build_curated_word.py constructs a
   fresh row/col permutation on demand for each letter (see
   `_permutation_for_trial()` -- any physical trial's real target can be
   relabeled as any desired character this way, no pre-curated variant
   set needed), but `LiveDecoder` has no idea any of that happened. It
   decodes a row/col through the standard, unpermuted
   `FARWELL_DONCHIN_GRID`. Comparing that straight against a
   permutation-relabeled character makes an otherwise-correct decode
   look wrong. Each letter's own `row_perm`/`col_perm` (returned by
   `build_word()`) is exactly the permutation that produced that letter's
   character -- `idx_to_char()` below translates a decoded index back
   through it before comparing. This was the actual cause of most
   "mismatches" seen in an earlier version of this cell: without this
   translation, a real run against "HELLO WORLD" looked like 1/11
   correct; with it, the same run is 11/11.
2. **Per-epoch visibility.** Rather than treat `LiveDecoder.run()` as a
   black box and only look at the final decoded letter, this replicates
   its loop manually (`get_epoch()` -> `score_epoch()` -> `should_decode()`,
   the same calls `run()` makes internally) so every epoch's stim_id,
   ground-truth target flag, classifier score, and current top candidate
   get logged via `TrainingLogger` (the same one scripts/train_offline.py
   uses) -- both to stdout and to a `run.log` file under `results/`, so a
   run can be reread afterward instead of re-executed to see where a
   particular letter went wrong.

One other subtlety: `LiveDecoder` is built once and reused across the
whole message -- its `stream` and `aligner` attributes are swapped before
each letter's epochs are fed in, since the loop reads both off `self`
rather than taking them as arguments. Only the first symbol crossing
threshold is kept per letter; a live system would move on to the next
letter as soon as one fires, so continuing to feed that letter's
remaining flashes into the same accumulator (which can trigger a second,
spurious decode) would be a replay artifact, not real behavior.

In [ ]:
import sys
import time
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent  # scripts/simulate_live/ -> repo root
sys.path.insert(0, str(REPO_ROOT / "scripts" / "curate"))
from build_curated_word import build_word

from eyecando.decode.lm import FARWELL_DONCHIN_GRID
from eyecando.live.decoder import LiveDecoder, PrebuiltEpochStream
from eyecando.pipeline.model import P300Model
from eyecando.decode.stopping import should_decode
from eyecando.utils.training_logger import TrainingLogger

MODEL_PATH = REPO_ROOT / "models" / "p300_classifier.pkl"
model = P300Model.load(MODEL_PATH)

MESSAGE = "ilajneksdlsemskld"  # edit this to whatever you want to spell/test


def idx_to_char(idx: int, row_perm: dict[int, int] | None, col_perm: dict[int, int] | None) -> str:
    """Undo the standard FARWELL_DONCHIN_GRID row-major mapping, apply
    this letter's own (row_perm, col_perm) if it has one, then re-map --
    the correct way to interpret a decoded index when the trial it came
    from was labeled via an on-demand permutation (see
    build_curated_word.py's _permutation_for_trial())."""
    row, col = idx // 6 + 1, idx % 6 + 7
    if row_perm is not None:
        row = row_perm[row]
    if col_perm is not None:
        col = col_perm[col]
    return FARWELL_DONCHIN_GRID[(row - 1) * 6 + (col - 7)]


run_dir = REPO_ROOT / "results" / f"session_runner_{time.strftime('%y%m%d-%H%M%S')}"
run_dir.mkdir(parents=True, exist_ok=True)
logger = TrainingLogger(run_dir / "run.log")
logger.log(f"[SESSION] message={MESSAGE.upper()!r} model={MODEL_PATH}")

letters = build_word(MESSAGE.upper())

decoded_chars = []
flashes_per_letter = []  # epochs actually consumed per letter -- for ITR's timing term
y_true_all = []  # every epoch's ground-truth is_target, across the whole message
y_score_all = []  # every epoch's classifier decision score, across the whole message
y_proba_all = []  # every epoch's predicted target-class probability, across the whole message
decoder = None
for i, letter in enumerate(letters):
    logger.log(
        f"[LETTER {i}] intended={letter['character']!r} "
        f"subject={letter['subject']} session={letter['session']} trial={letter['trial_index']} "
        f"n_calibration={letter['n_calibration_epochs']} (real={letter['n_real_epochs']}) "
        f"aligner={'ready' if letter['aligner'] is not None else 'NONE'}"
    )

    stream_items = [
        (int(stim_id), int(flag == 2), epoch[None, ...])
        for stim_id, flag, epoch in zip(
            letter["stim_ids"], letter["target_flags"], letter["epochs"]
        )
    ]
    stream = PrebuiltEpochStream(stream_items)
    if decoder is None:
        decoder = LiveDecoder(stream=stream, model=model, aligner=letter["aligner"])
    else:
        decoder.stream = stream
        decoder.aligner = letter["aligner"]

    decoded_symbol = None
    epoch_idx = 0
    while True:
        item = stream.get_epoch(timeout=decoder.get_epoch_timeout)
        if item is None:
            break
        stim_id, is_target, epoch = item
        # transform() is read-only (only update() mutates aligner state), so
        # calling it here before score_epoch() -- which transforms the same
        # epoch again internally -- gives an identical whitened epoch; this
        # is just the only way to get predict_proba() on the exact epoch
        # score_epoch() itself scored, without duplicating its logic.
        epoch_ea = decoder.aligner.transform(epoch)
        proba = float(model.predict_proba(epoch_ea)[0])
        y_proba_all.append(proba)

        score = decoder.score_epoch(stim_id, epoch)
        y_true_all.append(is_target)
        y_score_all.append(score)
        decode, idx = should_decode(
            decoder.accumulator,
            decoder.threshold,
            lm=decoder.lm,
            context=decoder.context,
            temperature=decoder.temperature,
            lm_temperature=decoder.lm_temperature,
        )
        top_char = idx_to_char(idx, letter["row_perm"], letter["col_perm"])
        logger.log(
            f"  [EPOCH {epoch_idx:02d}] stim_id={stim_id:2d} true_target={bool(is_target)} "
            f"score={score:+.3f} proba={proba:.3f} top_candidate={top_char!r} decode={decode}"
        )
        epoch_idx += 1
        if decode:
            decoded_symbol = idx
            if decoder.lm is not None:
                decoder.context += decoder.lm.character_set[idx]
            break
    stream.stop()
    flashes_per_letter.append(epoch_idx)

    decoded_char = (
        idx_to_char(decoded_symbol, letter["row_perm"], letter["col_perm"])
        if decoded_symbol is not None
        else "?"
    )
    match = "OK" if decoded_char == letter["character"] else "MISMATCH"
    logger.log(f"[LETTER {i}] decoded={decoded_char!r} intended={letter['character']!r} -- {match}")
    decoded_chars.append(decoded_char)

decoded_text = "".join(decoded_chars)
logger.log(f"[SESSION] intended={MESSAGE.upper()!r} decoded={decoded_text!r}")

print(f"Full log: {run_dir / 'run.log'}")
print(f"Intended: {MESSAGE.upper()!r}")
print(f"Decoded:  {decoded_text!r}")

### Diff against the intended message

In [ ]:
import difflib

for line in difflib.ndiff(MESSAGE.upper(), decoded_text):
    print(line)

n_correct = sum(a == b for a, b in zip(MESSAGE.upper(), decoded_text))
print()
print(f"{n_correct}/{len(MESSAGE)} letters correct")

### ITR and flash-level classification metrics

**Requires:** `decoded_text`, `n_correct`, `flashes_per_letter`,
`y_true_all`, `y_score_all`, `y_proba_all` from the session runner cell
above (and its diff cell for `n_correct`) -- run those first.

Three different things worth keeping separate:

- **Letter-level accuracy/ITR** -- did the *right symbol* get decoded,
  and how fast (in bits/min, via `compute_itr()`'s Wolpaw formula, the
  same one scripts/train_offline.py uses). `seconds_per_selection` here
  is the actual number of flashes this run consumed per letter (varies
  letter to letter -- some decode in one repetition, some exhaust all 96
  and never decode) times `SECONDS_PER_FLASH`, not an assumed fixed
  repetition count.
- **Flash-level classification metrics** -- accuracy/precision/recall/
  F1/AUC on every individual epoch's target-vs-nontarget call, pooled
  across the whole message. This is the same target:nontarget-imbalance
  situation `classifier.py`'s own `_classification_metrics` docstring
  describes (recall = catching real targets, precision structurally
  limited by the ~1:5/6 imbalance) -- it's a measure of the classifier
  itself, independent of whatever accumulation/threshold policy
  `should_decode` layers on top.
- **Bregman (beta-power) score** -- `scripts/simulate_dynamic_stopping.py`'s
  `beta_power_score`, a proper scoring rule (Bregman divergence of the
  convex generator phi(p)=p^beta) that dials between Brier (beta=2) and
  something closer to log loss's harshness on confidently-wrong
  predictions, while staying bounded in [0,1] -- see that script's own
  docstring for why beta=4 (`BETA_POWER_DEFAULT`) was chosen over plain
  Brier or log loss. Needs actual predicted probabilities, not decision
  scores, so the runner cell above also calls `model.predict_proba()` on
  each epoch's EA-whitened form (the same one `score_epoch()` scores,
  since `transform()` is read-only).

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

from eyecando.pipeline.classifier import N_SYMBOLS, SECONDS_PER_FLASH, compute_itr

sys.path.insert(0, str(REPO_ROOT / "scripts"))
from simulate_dynamic_stopping import BETA_POWER_DEFAULT, beta_power_score

letter_accuracy = n_correct / len(MESSAGE)
avg_flashes_per_letter = float(np.mean(flashes_per_letter))
seconds_per_selection = avg_flashes_per_letter * SECONDS_PER_FLASH
itr = compute_itr(letter_accuracy, N_SYMBOLS, seconds_per_selection)
total_seconds = sum(flashes_per_letter) * SECONDS_PER_FLASH

print(f"Letter accuracy: {n_correct}/{len(MESSAGE)} = {letter_accuracy:.1%}")
print(f"Avg flashes/letter: {avg_flashes_per_letter:.1f} ({seconds_per_selection:.2f}s/selection)")
print(f"Total time: {total_seconds:.1f}s across {len(MESSAGE)} letters")
print(f"ITR: {itr:.2f} bits/min")

y_true = np.array(y_true_all)
y_score = np.array(y_score_all)
y_proba = np.array(y_proba_all)
y_pred = (y_score >= 0).astype(int)  # decision_function threshold is 0, not 0.5

print()
print("Flash-level (target vs nontarget) classification, pooled across the whole message:")
print(f"  accuracy:  {accuracy_score(y_true, y_pred):.3f}")
print(f"  precision: {precision_score(y_true, y_pred, zero_division=0):.3f}")
print(f"  recall:    {recall_score(y_true, y_pred, zero_division=0):.3f}")
print(f"  f1:        {f1_score(y_true, y_pred, zero_division=0):.3f}")
print(f"  auc:       {roc_auc_score(y_true, y_score):.3f}")
print(f"  bregman (beta={BETA_POWER_DEFAULT}): {beta_power_score(y_proba, y_true):.4f}  (0=perfect, proper scoring rule)")

## Live Test Segment B: flash/marker generator + marker-only validator

Builds and validates just the flash-sequencing and marker-push logic --
no rendering, no EEG, no `LiveDecoder` (see
`text_resources/live_test_segments_B_C_scope.md`). Unlike every other
script here (`simulate_lsl_outlet.py`, `curate_trial_epochs.py`), which
only replays a precomputed schedule whose timestamps were already known
in advance, `simulate_flash_sequencer.py` generates a genuinely live
sequence -- rows 1-6 and cols 7-12, freshly shuffled every round, one real
LSL marker pushed per flash as it's decided.

`consume_flash_markers.py` validates the result purely from the marker
stream's own content -- stim_id validity, per-round target/nontarget
structure, ISI timing, and the spelled sequence inferred from which
(row, col) each round marks as target -- independent of the generator's
own `--word` argument. Reuses `BackgroundScript` from above, same as the
outlet/consumer/trigger demo.

In [ ]:
segment_b_flash = BackgroundScript(
    [
        "scripts/simulate_live/simulate_flash_sequencer.py",
        "--word", "HI",
        "--rounds", "2",
        "--wait-for-trigger",
    ]
)
print(f"Flash sequencer running: {segment_b_flash.is_running()}")

segment_b_consumer = BackgroundScript(
    [
        "scripts/simulate_live/consume_flash_markers.py",
        "--quiet-timeout", "5",
        "--wait-for-trigger",
    ]
)
print(f"Marker validator running: {segment_b_consumer.is_running()}")

In [ ]:
# Same trigger_lsl_start.py used everywhere else in this notebook --
# unchanged. Blocks until both --wait-for-trigger processes above have
# attached to EyeCanDoControl, then re-pushes over --push-duration to
# catch both.
result = subprocess.run(
    [sys.executable, "scripts/simulate_live/trigger_lsl_start.py"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout, end="")
print(result.stderr, end="")
result.check_returncode()

# Poll until both finish rather than a single fixed sleep -- process
# startup/import overhead varies enough that a fixed guess is flaky
# (confirmed directly). 2 characters x 2 rounds x 12 flashes/round x
# 0.25s ISI = ~12s streaming, plus the validator's own --quiet-timeout
# after that.
max_wait = 60.0
poll_interval = 1.0
waited = 0.0
while (segment_b_flash.is_running() or segment_b_consumer.is_running()) and waited < max_wait:
    time.sleep(poll_interval)
    waited += poll_interval
print(f"Done waiting after {waited:.0f}s.")

In [ ]:
print("--- flash sequencer ---")
print(segment_b_flash.output())
print("--- marker validator ---")
print(segment_b_consumer.output())

In [ ]:
segment_b_flash.is_running(), segment_b_consumer.is_running()

## Live Test Segment C: live markers against a streamed EEG source

Segment B (scripts/simulate_live/simulate_flash_sequencer.py + scripts/simulate_live/consume_flash_markers.py,
run standalone in two terminals) already proved the marker generator itself
is correct and well-formed, independent of everything downstream. This
segment connects that live generator to a real `LSLEEGStream` -- paired
with EEG replayed via `simulate_lsl_outlet.py`, since there's no real
headset to drive EEG from the actual flashes the generator is producing.

**What this does and doesn't prove** (see
`text_resources/live_test_segments_B_C_scope.md`): the EEG has no causal
relationship to whichever flash the live generator calls "target" -- no
real P300 exists at those moments, because nothing attended them. This
segment cannot and does not claim anything about decode correctness (that's
already covered separately by the curated-word session runner above, using
real recorded target trials). What it actually checks is plumbing: does
`LSLEEGStream`'s marker-reader/epoch-builder handle genuinely
live-arriving markers -- real wall-clock jitter, two independently-clocked
streams -- without crashing or silently dropping, with the right epoch
count and sensible `max_marker_lag` behavior.

Reuses the `BackgroundScript` class from above (no changes needed) to
launch `simulate_lsl_outlet.py` (replayed EEG) and
`simulate_flash_sequencer.py` (live markers) together, `trigger_lsl_start.py`
(unchanged) to synchronize both starts, and `LSLEEGStream` directly (also
unchanged -- it already handles exactly this scenario).

In [ ]:
import mne
import warnings

from eyecando.ingestion.data import load_moabb_session
from eyecando.live.streaming import LSLEEGStream, iter_epochs

mne.set_log_level("WARNING")

EEG_STREAM_NAME = "EyeCanDoEEG"
MARKER_STREAM_NAME = "EyeCanDoMarkers"

# Same subject/session simulate_lsl_outlet.py will replay -- only used here
# to learn the real channel names/sample rate LSLEEGStream needs to match.

_raw = load_moabb_session(subject=1, session=0)
_eeg_picks = mne.pick_types(_raw.info, eeg=True)
ch_names = [_raw.ch_names[i] for i in _eeg_picks]
sfreq = _raw.info["sfreq"]

segment_c_eeg = BackgroundScript(
    [
        "scripts/simulate_live/simulate_lsl_outlet.py",
        "--subject", "1",
        "--session", "0",
        "--wait-for-trigger",
    ]
)
print(f"EEG outlet running: {segment_c_eeg.is_running()}")

segment_c_flash = BackgroundScript(
    [
        "scripts/simulate_live/simulate_flash_sequencer.py",
        "--word", "HELLO",
        "--rounds", "2",
        "--wait-for-trigger",
    ]
)
print(f"Flash sequencer running: {segment_c_flash.is_running()}")

In [ ]:
# Plain resolve+connect -- no --wait-for-trigger logic needed on this side,
# since both outlets above already gate on wait_for_consumers()/the shared
# trigger before producing anything; LSLEEGStream.start() just needs both
# streams to exist by the time it resolves them (see resolve_timeout).
segment_c_stream = LSLEEGStream(
    eeg_stream_name=EEG_STREAM_NAME,
    marker_stream_name=MARKER_STREAM_NAME,
    sfreq=sfreq,
    ch_names=ch_names,
    resolve_timeout=30.0,
)
print("Resolving streams (this blocks until both outlets above are up)...")
segment_c_stream.start()
print("Resolved and started.")

In [ ]:
# Same trigger_lsl_start.py used for the outlet/consumer workflow above --
# unchanged. Blocks until both --wait-for-trigger processes have attached
# to EyeCanDoControl, then re-pushes over --push-duration to catch both.
result = subprocess.run(
    [sys.executable, "scripts/simulate_live/trigger_lsl_start.py"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout, end="")
print(result.stderr, end="")
result.check_returncode()

In [ ]:
print("--- EEG outlet ---")
print(segment_c_eeg.output())
print("--- flash sequencer ---")
print(segment_c_flash.output())

In [ ]:
# Pull epochs until the stream goes quiet for `timeout` seconds -- the
# same "no epoch within timeout -- assume the replay finished" pattern
# consume_lsl_stream.py uses. Capture any max_marker_lag-triggered drop
# warnings (see streaming.py::_process_marker) instead of letting them
# just print, so the summary below can report them explicitly.
n_epochs = 0
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    for stim_id, is_target, epoch in iter_epochs(segment_c_stream, timeout=10.0):
        n_epochs += 1

drop_warnings = [str(w.message) for w in caught if "dropped" in str(w.message)]
print(f"Epochs received: {n_epochs}")
print(f"Dropped-marker warnings: {len(drop_warnings)}")
for msg in drop_warnings:
    print(f"  {msg}")

In [ ]:
import re

# The sequencer's own end-of-run summary line is the ground truth for how
# many flashes it actually pushed -- e.g. "done streaming ('HELLO' spelled,
# 120 flashes pushed, 30.1s elapsed)".
match = re.search(r"(\d+) flashes pushed", segment_c_flash.output())
expected_flashes = int(match.group(1)) if match else None
print(f"Sequencer reported: {expected_flashes} flashes pushed")
print(f"LSLEEGStream received: {n_epochs} epochs")

if expected_flashes is None:
    print("FAIL -- could not parse the sequencer's flash count from its output")
elif n_epochs == expected_flashes:
    print("PASS -- epoch count matches flashes pushed exactly")
elif drop_warnings and n_epochs == expected_flashes - len(drop_warnings):
    print("PASS -- epoch count matches after accounting for logged max_marker_lag drops")
else:
    print("FAIL -- epoch count does not match flashes pushed, and not explained by logged drops")

# Stop the stream before the background processes, so this comparison
# isn't racing a still-running producer. simulate_lsl_outlet.py replays a
# whole session's worth of EEG -- much longer than this short flash
# sequence -- so it's still running at this point; give it a generous
# timeout, and don't let a slow-to-terminate subprocess (BackgroundScript's
# own kill()-then-wait() has no fallback if that second wait() also times
# out) crash this cell.
segment_c_stream.stop()


def _safe_stop(bg: "BackgroundScript", name: str) -> None:
    try:
        bg.stop(timeout=15.0)
    except subprocess.TimeoutExpired:
        print(f"  warning: {name} did not terminate cleanly within 15s")


_safe_stop(segment_c_eeg, "EEG outlet")
_safe_stop(segment_c_flash, "flash sequencer")
print(
    f"Stopped. eeg running={segment_c_eeg.is_running()} "
    f"flash running={segment_c_flash.is_running()}"
)